In [1]:
from sklearn.model_selection import KFold
from sklearn.model_selection import train_test_split
from matplotlib import pyplot as plt
import random
import tensorflow as tf
import numpy as np
import timeit
from sklearn import metrics
import scipy.io as sio
import csv
import os
import math 
import random as rand
from numpy import genfromtxt
from numpy import sum,isrealobj,sqrt
from numpy.random import standard_normal

#import torch
#from torch import nn

import warnings
from scipy.spatial import distance



import sys
sys.path.append("..") # Adds higher directory to python modules path.

from Modules.Tools import MapTool
from Modules.Tools import BoxPlot
from Modules.Tools import peakCorrelation as pC

In [2]:
# Signal 
params = {'legend.fontsize': 50,
          'figure.figsize': (45, 15),
         'axes.labelsize': 60,
         'axes.titlesize':70,
         'xtick.labelsize':50,
         'ytick.labelsize':50,
         'lines.linewidth': 8}
plt.rcParams.update(params)

# Notebook Parameters
- this is simply for obtaining the graphs with manual mapping. Pad is closely relate to the sampling rate for the EM signal

In [3]:
pad = 8

# Create the code.

In [4]:
Alpha_sub_Base_Code = np.array(["ori", "sts", "lds", "ori", "mov",
         # Injected Code
         "sub",                   
        "andi", "mov", "ori", "mov", "andi"])

# Fill with Nops before.

In [9]:
'''
Simply adds the number of nops to the array
'''
def addNops(number, array):
    code_list = []
    count = 0
    while count < number:
        code_list.append('nop')
        count = count+1
    
    for code in array:
        code_list.append(code)
    return np.array(code_list)

## Test

In [ ]:
print(addNops(4, Alpha_sub_Base_Code))

## Create

In [11]:
Alpha_sub_Code_5_Seq = addNops(4, Alpha_sub_Base_Code)

# Split code into cycles.

In [16]:
'''
Simply splits known multi cycle codes.
'''
def splitCode(array):
    code_list = []
    two_cycle_list = ["sts", "lds", "muls", "fmul", "push", "pop"]
    three_cycle_list = ["jmp"]
    for code in array:
        # check for hyphens
        if code == "brne-f":
            code_list.append("brneF")
        elif code == "brne-t":
            code_list.append("brneTPT1")
            code_list.append("brneTPT2")
        elif code == "brpl-f":
            code_list.append("brplF")
        elif code == "sbis-f":
            code_list.append("sbisF")
        
        # check for two cycles
        elif code in two_cycle_list:
            code_list.append(code + "PT1")
            code_list.append(code + "PT2")
            
        # check for thee cycles
        elif code in three_cycle_list:
            code_list.append(code + "PT1")
            code_list.append(code + "PT2")
            code_list.append(code + "PT3")
        
        else:
            code_list.append(code)
            
    return np.array(code_list)

## Test

In [ ]:
print(splitCode(Alpha_sub_Code_5_Seq))

## Create

In [18]:
Alpha_sub_Code_5_Seq_Split = splitCode(Alpha_sub_Code_5_Seq)

# Create 5 sequence of instructions for the codes

In [29]:
def createBaseCode(code):
    seq_num = 5
    for n in range(code.shape[0]-seq_num+1):
        seq = []
        for q in range(seq_num):
            seq.append(code[n+q])
        print(str(n) +": "+ str(seq))

## Test

## Alpha

In [ ]:
createBaseCode(Alpha_sub_Code_5_Seq_Split)

## Also need to tokenize via the same tokenizer for the code.

In [35]:
from tensorflow.keras.preprocessing.text import Tokenizer

In [36]:
load_Dataset_Folder = "./Datasets/FP_GANs_LargeScale_2/k-Fold/TrainingDatasets/"
All_Code = np.load(load_Dataset_Folder + "All_Code.npy", allow_pickle = True)

tokenizer = Tokenizer(num_words=100, char_level=False)

all_instructions = All_Code[0] 

sentence_Code = [all_instructions]
tokenizer.fit_on_texts(sentence_Code)

print(sentence_Code)
print(tokenizer.texts_to_sequences(sentence_Code))

['add, and, andi, brneF, brneTPT1, brneTPT2, brplF, bset, cli, cln, com, cp, dec, eor, fmulPT1, fmulPT2, in, inc, jmpPT1, jmpPT2, jmpPT3, ldi, ldsPT1, ldsPT2, mov, mulsPT1, mulsPT2, neg, nop, or, ori, out, popPT1, popPT2, pushPT1, pushPT2, sbc, sbci, sbisF, sen, sev, stsPT1, stsPT2, sub, swap, tst']
[[1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19, 20, 21, 22, 23, 24, 25, 26, 27, 28, 29, 30, 31, 32, 33, 34, 35, 36, 37, 38, 39, 40, 41, 42, 43, 44, 45, 46]]


In [38]:
print(tokenizer.texts_to_sequences(['sev']))

[[41]]


In [47]:
def createBaseCode(code):
    Base = []
    seq_num = 5
    for n in range(code.shape[0]-seq_num+1):
        seq = []
        for q in range(seq_num):
            seq.append(code[n+q])
        #print(seq)
        tokens = tokenizer.texts_to_sequences(seq)
        #print(tokens)
        token_list = []
        for token in tokens:
            token_list.append(token[0])
        Base.append(token_list)
        #print(Base)
    return np.array(Base)

## Test

In [56]:
createBaseCode(Alpha_sub_Code_5_Seq_Split)

array([[29, 29, 29, 29, 31],
       [29, 29, 29, 31, 42],
       [29, 29, 31, 42, 43],
       [29, 31, 42, 43, 23],
       [31, 42, 43, 23, 24],
       [42, 43, 23, 24, 31],
       [43, 23, 24, 31, 25],
       [23, 24, 31, 25, 44],
       [24, 31, 25, 44,  3],
       [31, 25, 44,  3, 25],
       [25, 44,  3, 25, 31],
       [44,  3, 25, 31, 25],
       [ 3, 25, 31, 25,  3]])

## Create

In [50]:
Alpha_sub_Base_Code = createBaseCode(Alpha_sub_Code_5_Seq_Split)

# Save to utalize later

In [59]:
np.save(".\Datasets\FP_GANs_LargeScale_2\k-Fold\TrainingDatasets\Alpha_sub_Base_code.npy", Alpha_sub_Base_Code)